# UniText — Python quickstart

`unitext` is a Cython extension over the UniText C ABI, shipped as a
self-contained wheel: the native library travels inside the package, so
installing it needs neither Nim nor a compiler.

```
pip install unitext
```

CI executes this notebook against the wheel the release actually publishes, so
an output below that stops matching fails the build.

## The API

In [1]:
import unitext

unitext.version(), unitext.__version__

('0.1.0', '0.1.0')

`fibonacci` is the template's hello-world, iterative and O(n).

In [2]:
[unitext.fibonacci(n) for n in range(11)]

[0, 1, 1, 2, 3, 5, 8, 13, 21, 34, 55]

## The domain is part of the contract

`fibonacci` is defined on `[0, 92]` — 92 being the largest argument whose result
still fits in a signed 64-bit integer. The bound is not advisory.

In [3]:
unitext.fibonacci(92)

7540113804746346429

Past it the binding raises, rather than returning a silently wrong
number. This is the contract the Nim library states as a precondition; each
surface expresses it in the terms its own callers expect.

In [4]:
try:
    unitext.fibonacci(93)
except ValueError as exc:
    print("ValueError:", exc)

ValueError: n must be in [0, 92], got 93


In [5]:
try:
    unitext.fibonacci(-1)
except ValueError as exc:
    print("ValueError:", exc)

ValueError: n must be in [0, 92], got -1


A non-integer argument is a type error, not a coercion.

In [6]:
try:
    unitext.fibonacci(10.0)
except TypeError as exc:
    print("TypeError:", exc)

TypeError: n must be int, got float


## The C ABI underneath

The same entry points are reachable from anything that speaks C. There the
contract is expressed by clamping instead of raising — an exception must never
unwind across an ABI boundary:

```c
unitext_fibonacci(-5);   /* 0       — clamped */
unitext_fibonacci(200);  /* fib(92) — clamped */
```

See `include/UniText.h`, and the book for the full picture.